# Cobertura mediática del agua en la ZMG · Colab + Google Sheets

Corre el scraper en Colab y usa **tu Google Sheet** como panel: ahí defines los medios y las búsquedas, ahí corriges
la postura de las notas y ahí se escriben los resultados.

**Orden:** 1) instalar → 2) parámetros → 3) conectar la hoja → 4) recolectar → 5) descargar → 6) analizar → 7) exportar.
Si se corta la sesión, vuelve a correr desde el paso 1: con `GUARDAR_EN_DRIVE = True` retoma donde se quedó.

## 1 · Instalar y traer el código

In [ ]:
!pip -q install feedparser PyYAML Jinja2 lxml beautifulsoup4 requests gspread googlenewsdecoder

import os, sys, subprocess
REPO = "rogealdata/agua-gdl"   # tu_usuario/nombre_del_repo
RAMA = "main"
if not os.path.exists("/content/agua-gdl"):
    try:   # si el repo es privado, guarda un token en Colab: 🔑 Secrets -> GITHUB_TOKEN
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = None
    url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
    subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/agua-gdl"], check=True)
sys.path.insert(0, "/content/agua-gdl")
print("código listo")

## 2 · Parámetros

In [ ]:
HOJA = ""              #@param {type:"string"}   URL de tu Google Sheet. Vacío = crea una nueva
DESDE = "2025-01-01"   #@param {type:"date"}     fecha mínima de las notas
MODOS = ["rss", "sitemap", "google"]  # cómo descubrir notas: RSS del medio, sitemap del medio y/o Google Noticias
MAX_MEDIOS = 0         #@param {type:"integer"}  0 = todos los activos; N = solo los primeros N de la pestaña Medios
SOLO_MEDIOS = ""       #@param {type:"string"}   ids separados por coma (p. ej. informador,jornada). Vacío = todos
PAUSA = 1.5            #@param {type:"number"}   segundos entre peticiones al mismo sitio
LIMITE_DESCARGA = 0    #@param {type:"integer"}  0 = sin límite (útil para una prueba rápida: 30)
GUARDAR_EN_DRIVE = True  #@param {type:"boolean"} guarda la base en Drive para retomar y acumular corridas

## 3 · Conectar con Google Sheets

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
auth.authenticate_user()
gc = gspread.authorize(default()[0])

from agua_gdl.config import Config
from agua_gdl import sheets

cfg = Config()
sh = gc.open_by_url(HOJA) if HOJA else gc.create("Agua GDL · cobertura mediática")
sheets.seed_config(sh, cfg)          # crea Medios / Busquedas / Revision si no existen
print("Hoja:", sh.url)
print("→ Revisa/edita las pestañas Medios y Busquedas y luego sigue. En Medios, activo = no apaga un medio.")

In [ ]:
cfg = sheets.apply_config(sh, Config(), max_medios=MAX_MEDIOS or None)
if SOLO_MEDIOS.strip():
    ids = {x.strip() for x in SOLO_MEDIOS.split(",")}
    cfg.sources = [s for s in cfg.sources if s["id"] in ids]
print(f"{len(cfg.sources)} medios · {len(cfg.keywords['busquedas'])} búsquedas · modos: {MODOS}")
for s in cfg.sources: print(f"  {s['scope']:9} {s['id']:14} {s['domain']}")

from agua_gdl.db import DB
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/agua-gdl", exist_ok=True)
    DB_PATH = "/content/drive/MyDrive/agua-gdl/agua.db"
else:
    DB_PATH = "/content/agua.db"
db = DB(DB_PATH)

## 4 · Recolectar notas
Descubre URLs. **RSS/sitemap** raspan directo los sitios; **google** busca por dominio en Google Noticias.
Los fallos por fuente se reportan y no detienen el proceso.

In [ ]:
from agua_gdl.collect import collect
from agua_gdl.http import Client

stats = collect(cfg, db, Client(delay=PAUSA), DESDE, modes=tuple(MODOS))
print(f"\n{stats['nuevas']} notas nuevas de {stats['vistas']} vistas · {len(stats['fallas'])} fuentes con falla")

## 5 · Descargar el texto de cada nota
Puede tardar (≈ PAUSA segundos por nota). Si se interrumpe, vuelve a correr esta celda.

In [ ]:
from agua_gdl.fetch import fetch_all
fetch_all(db, Client(delay=PAUSA), limit=LIMITE_DESCARGA or None)

## 6 · Analizar
Aplica las correcciones de la pestaña **Revision** (columnas `url`, `postura` = `a_favor|en_contra|mixta|neutral`).

In [ ]:
from agua_gdl.pipeline import analyze_all
analyze_all(db, cfg, overrides=sheets.read_overrides(sh))

## 7 · Exportar a la hoja y generar el reporte HTML

In [ ]:
res = sheets.export(sh, db, cfg.themes)
print(res, "\n", sh.url)

from pathlib import Path
from agua_gdl.report import build
out = Path("/content/drive/MyDrive/agua-gdl/reporte") if GUARDAR_EN_DRIVE else Path("/content/reporte")
print(build(db, cfg.themes, out, Path("/content/agua-gdl/agua_gdl/templates")))
print("Reporte HTML y CSVs en:", out)

### Cómo corregir la clasificación
1. En la pestaña **En contra** o **A favor** encuentra una nota mal clasificada y revisa las columnas de *evidencia*.
2. Pega su `url` en la pestaña **Revision** con la postura correcta.
3. Vuelve a correr los pasos 6 y 7. La nota queda marcada como corregida a mano.
4. Para mejorar el léxico en general, edita `config/lexicon.yaml` del repo (ver README).